In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
df = pd.read_csv("ecommerce_behavior.csv")
print(df.shape)
print(df.dtypes)
df.head()

In [ ]:
# Drop nulls
df.dropna(inplace=True)

# Fix event_time to datetime
df["event_time"] = pd.to_datetime(df["event_time"])

# Extract time features
df["Month"] = df["event_time"].dt.to_period("M")
df["Hour"] = df["event_time"].dt.hour

print(df["event_type"].value_counts())

In [ ]:
# Count each funnel stage
funnel = df["event_type"].value_counts().reindex(["view", "cart", "purchase"])
funnel = funnel.reset_index()
funnel.columns = ["Stage", "Count"]

# Calculate conversion rates
funnel["Conversion Rate (%)"] = (funnel["Count"] / funnel["Count"].iloc[0] * 100).round(2)

print(funnel)

In [ ]:
colors = ["steelblue", "orange", "green"]

plt.figure(figsize=(8, 5))
bars = plt.barh(funnel["Stage"], funnel["Count"], color=colors)
plt.xlabel("Number of Events")
plt.title("Marketing Funnel: View → Cart → Purchase")
plt.gca().invert_yaxis()

# Add count labels
for bar, val in zip(bars, funnel["Count"]):
    plt.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va="center", fontsize=10)

plt.tight_layout()
plt.savefig("funnel_chart.png")
plt.show()

In [ ]:
view_to_cart = (funnel.loc[funnel["Stage"]=="cart", "Count"].values[0] /
                funnel.loc[funnel["Stage"]=="view", "Count"].values[0] * 100)

cart_to_purchase = (funnel.loc[funnel["Stage"]=="purchase", "Count"].values[0] /
                    funnel.loc[funnel["Stage"]=="cart", "Count"].values[0] * 100)

view_to_purchase = (funnel.loc[funnel["Stage"]=="purchase", "Count"].values[0] /
                    funnel.loc[funnel["Stage"]=="view", "Count"].values[0] * 100)

print(f"View → Cart Conversion Rate:     {view_to_cart:.2f}%")
print(f"Cart → Purchase Conversion Rate: {cart_to_purchase:.2f}%")
print(f"Overall View → Purchase Rate:    {view_to_purchase:.2f}%")

In [ ]:
top_categories = df[df["event_type"] == "view"]["category_code"].value_counts().nlargest(10).reset_index()
top_categories.columns = ["Category", "Views"]

sns.barplot(data=top_categories, x="Views", y="Category", palette="Blues_r")
plt.title("Top 10 Product Categories by Views")
plt.xlabel("Total Views")
plt.ylabel("")
plt.tight_layout()
plt.savefig("top_categories_views.png")
plt.show()

In [ ]:
top_purchases = df[df["event_type"] == "purchase"]["category_code"].value_counts().nlargest(10).reset_index()
top_purchases.columns = ["Category", "Purchases"]

sns.barplot(data=top_purchases, x="Purchases", y="Category", palette="Greens_r")
plt.title("Top 10 Product Categories by Purchases")
plt.xlabel("Total Purchases")
plt.ylabel("")
plt.tight_layout()
plt.savefig("top_categories_purchases.png")
plt.show()

In [ ]:
hourly = df.groupby(["Hour", "event_type"]).size().reset_index(name="Count")
hourly = hourly[hourly["event_type"].isin(["view", "purchase"])]

sns.lineplot(data=hourly, x="Hour", y="Count", hue="event_type", palette={"view": "steelblue", "purchase": "green"})
plt.title("Hourly Traffic vs Purchase Patterns")
plt.xlabel("Hour of Day")
plt.ylabel("Event Count")
plt.tight_layout()
plt.savefig("hourly_patterns.png")
plt.show()

In [ ]:
monthly_purchases = df[df["event_type"] == "purchase"].groupby("Month").size().reset_index(name="Purchases")
monthly_purchases["Month"] = monthly_purchases["Month"].astype(str)

plt.plot(monthly_purchases["Month"], monthly_purchases["Purchases"], marker="o", color="green")
plt.xticks(rotation=45)
plt.title("Monthly Purchase Trend")
plt.xlabel("Month")
plt.ylabel("Number of Purchases")
plt.tight_layout()
plt.savefig("monthly_purchases.png")
plt.show()

## Key Business Insights

1. **Funnel Drop-off**: The biggest drop occurs between View and Cart,
   meaning most users browse but don't add items to cart.
2. **Cart Abandonment**: A significant portion of users who add to cart
   don't complete the purchase — a major revenue leak.
3. **Top Categories**: Electronics and smartphones dominate both 
   views and purchases.
4. **Peak Hours**: Traffic and purchases peak in the evening (6PM–9PM),
   suggesting the best time to run promotions.
5. **Monthly Trend**: Purchase volumes show growth over time with 
   noticeable seasonal spikes.

## Recommendations
- Add exit-intent popups or discounts to convert browsers into cart adders.
- Send cart abandonment emails or push notifications to recover lost sales.
- Focus ad spend on top-performing categories like electronics.
- Schedule promotions and flash sales during peak evening hours.
- Plan inventory and campaigns around seasonal purchase spikes.